In [25]:
import requests
import time
import random
from bs4 import BeautifulSoup
import polars as pl
import re
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass
import pandas as pd


In [17]:
BASE = "http://dictybase.org"

# get curator notes

In [33]:
# good for large qureies
session = requests.Session()
session.headers.update({
    "User-Agent": "dictybase-curator-notes/0.1"
})


In [58]:
def get_curator_notes_html(gene_id: str, timeout: float = 15.0) -> str | None:
    """
    Return curator notes as HTML-ish string (with <i>, <br>, etc.),
    or None if 404 / no notes.
    """
    url = f"{BASE}/gene/{gene_id}/gene/summary.json"
    r = session.get(url, timeout=timeout)

    if r.status_code == 404:
        return None
    r.raise_for_status()

    data = r.json()

    try:
        col0 = data[0]["items"][0]
        col_items = col0["content"][0]["items"]
        content_row = col_items[1]                  # after "Curator Notes" title
        tokens = content_row["content"][0]["items"]
    except (KeyError, IndexError, TypeError):
        return None

    fragments = []
    for t in tokens:
        if "text" in t:
            fragments.append(t["text"])
        elif "caption" in t:
            fragments.append(t["caption"])

    html = "".join(fragments).strip()
    return html or None


def get_curator_notes_plain(gene_id: str, timeout: float = 15.0) -> str | None:
    """
    Plain-text version of curator notes (HTML stripped).
    """
    html = get_curator_notes_html(gene_id, timeout=timeout)
    if html is None:
        return None

    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(" ", strip=True)

In [64]:
def get_curator_notes_tokens(gene_id: str, timeout: float = 15.0):
    url = f"{BASE}/gene/{gene_id}/gene/summary.json"
    r = session.get(url, timeout=timeout)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    data = r.json()

    try:
        col0 = data[0]["items"][0]
        col_items = col0["content"][0]["items"]
        content_row = col_items[1]
        tokens = content_row["content"][0]["items"]
    except (KeyError, IndexError, TypeError):
        return None

    return [t for t in tokens if isinstance(t, dict)]


In [65]:
SENT_TOKEN_PAT = re.compile(r"\([^)]*\)|\[\[PUB:(\d+)\]\]")  # parenthetical OR marker

def _clean_text(s: str) -> str:
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    return s

def _build_claim_and_anchors_from_marker_sentence(sent_markers: str):
    """
    From a sentence containing [[PUB:####]] markers:
      - remove citation parentheticals (those containing markers)
      - remove bare markers
      - record anchor positions (char offsets) in the cleaned claim text
    """
    raw_claim_parts = []
    anchors_raw = {}  # pos_raw -> [pub_ids]
    i = 0

    for m in SENT_TOKEN_PAT.finditer(sent_markers):
        start, end = m.start(), m.end()
        before = sent_markers[i:start]
        raw_claim_parts.append(before)

        chunk = sent_markers[start:end]

        # parenthetical
        if chunk.startswith("(") and chunk.endswith(")"):
            pub_ids = PUB_MARKER_PAT.findall(chunk)
            if pub_ids:
                pos_raw = len("".join(raw_claim_parts))
                anchors_raw.setdefault(pos_raw, []).extend(pub_ids)
                # drop the entire citation parenthetical
            else:
                raw_claim_parts.append(chunk)  # keep non-citation parentheses
        else:
            # bare marker [[PUB:####]]
            pid = m.group(1)
            if pid:
                pos_raw = len("".join(raw_claim_parts))
                anchors_raw.setdefault(pos_raw, []).append(pid)
            # drop the marker

        i = end

    raw_claim_parts.append(sent_markers[i:])
    raw_claim = "".join(raw_claim_parts)

    claim_plain = _clean_text(raw_claim)

    # remap raw anchor positions -> cleaned-text positions
    anchors = []
    for pos_raw, pids in anchors_raw.items():
        prefix_clean = _clean_text(raw_claim[:pos_raw])
        pos_clean = len(prefix_clean)
        anchors.append({
            "pos": pos_clean,
            "pub_ids": [int(x) for x in _dedup_keep_order(pids)],
        })

    anchors = sorted(anchors, key=lambda d: d["pos"])
    return claim_plain, anchors


In [76]:

YEAR_PAT = re.compile(r"\b(18\d{2}|19\d{2}|20\d{2})\b")
PUB_URL_PAT = re.compile(r"^/publication/(\d+)\b")
PUB_MARKER_PAT = re.compile(r"\[\[PUB:(\d+)\]\]")

def _strip_html_to_plain(s: str) -> str:
    return BeautifulSoup(s, "html.parser").get_text(" ", strip=True)

def _dedup_keep_order(xs):
    seen = set()
    out = []
    for x in xs:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out

def _is_citation_only_sentence(sent: str) -> bool:
    """
    True if sentence contains PUB markers but otherwise has no 'real' content
    (only parentheses/punctuation/spaces).
    """
    if not PUB_MARKER_PAT.search(sent):
        return False

    # Remove markers, then check what's left
    s = PUB_MARKER_PAT.sub("", sent)
    # Remove punctuation/whitespace
    s = re.sub(r"[\s\(\)\[\]\{\},;:.!?\-–—]+", "", s)
    return s == ""  # nothing meaningful left

def _remove_citations_from_sentence(sent_with_markers: str) -> str:
    """
    Remove citation markers and also remove citation-only parentheticals.
    Keep other parentheticals like (pkaC, pkaR).
    """
    def repl_paren(m):
        block = m.group(0)  # "( ... )"
        inner = block[1:-1]
        if not PUB_MARKER_PAT.search(inner):
            return block  # not a citation paren
        # remove markers
        inner2 = PUB_MARKER_PAT.sub("", inner)
        # if only punctuation/whitespace remains, drop entire parentheses
        inner2_chk = re.sub(r"[\s,;:.]+", "", inner2)
        if inner2_chk == "":
            return ""
        # otherwise keep remaining text (rare "see ..." cases)
        inner2 = inner2.strip()
        return f"({inner2})"

    # 1) first handle parenthetical blocks that contain markers
    s = re.sub(r"\([^)]*\)", repl_paren, sent_with_markers)

    # 2) remove any remaining markers not inside parentheses
    s = PUB_MARKER_PAT.sub("", s)

    # 3) cleanup spacing
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    return s

def extract_publication_claims_from_tokens(gene_id: str, tokens):
    if not tokens:   # None or empty list
        return [], []
    parts = []
    pub_meta = {}  # pub_id -> {caption_plain, year}

    for t in tokens:
        if "text" in t:
            parts.append(str(t["text"]))
            continue

        caption = t.get("caption")
        url = t.get("url")
        if caption is None:
            continue

        caption_str = str(caption)
        m = PUB_URL_PAT.match(str(url)) if url else None

        if m:
            pub_id = m.group(1)
            parts.append(f"[[PUB:{pub_id}]]")

            cap_plain = _strip_html_to_plain(caption_str)
            years = YEAR_PAT.findall(cap_plain)
            year = int(years[-1]) if years else None
            pub_meta[pub_id] = {"caption_plain": cap_plain, "year": year}
        else:
            parts.append(caption_str)

    html_with_markers = "".join(parts)

    tmp_plain = _strip_html_to_plain(
        html_with_markers.replace("<br>", ". ").replace("<br/>", ". ")
    )

    raw = [s.strip() for s in re.split(r"(?<=[.!?])\s+", tmp_plain) if s.strip()]

    merged = []
    for s in raw:
        if merged and _is_citation_only_sentence(s):
            merged[-1] = (merged[-1] + " " + s).strip()
        else:
            merged.append(s)

    claims = []
    for sent_markers in merged:
        pub_ids = PUB_MARKER_PAT.findall(sent_markers)
        if not pub_ids:
            continue

        pub_ids_u = _dedup_keep_order(pub_ids)

        # anchors + claim_plain (no citations)
        claim_plain, anchors = _build_claim_and_anchors_from_marker_sentence(sent_markers)

        # sentence_plain (citations rendered), plus a marked version
        sent_plain = sent_markers
        cited_sentence_marked = sent_markers

        caps = []
        years = []
        for pid in pub_ids_u:
            meta = pub_meta.get(pid, {})
            cap_plain = meta.get("caption_plain", f"publication/{pid}")

            sent_plain = sent_plain.replace(f"[[PUB:{pid}]]", cap_plain)
            cited_sentence_marked = cited_sentence_marked.replace(f"[[PUB:{pid}]]", f"[CITE:{pid}]")

            caps.append(cap_plain)
            if meta.get("year") is not None:
                years.append(int(meta["year"]))

        claims.append({
            "gene_id": gene_id,
            "sentence_markers": sent_markers,                 # keep original marker placement
            "sentence_plain": sent_plain,                     # readable with author-year
            "cited_sentence_marked": cited_sentence_marked,   # [CITE:####] markers
            "claim_plain": claim_plain,                       # claim only
            "anchors": anchors,                               # [{"pos": int, "pub_ids":[...]}]
            "publication_ids": [int(x) for x in pub_ids_u],
            "citation_captions": _dedup_keep_order(caps),
            "citation_years": _dedup_keep_order(years),
        })

    # publication table rows (dedup later)
    pub_rows = []
    for pid, meta in pub_meta.items():
        pub_rows.append({
            "publication_id": int(pid),
            "caption_plain": meta.get("caption_plain"),
            "year": meta.get("year"),
        })

    return claims, pub_rows



In [70]:
# Example
gid = "DDB_G0283907"
print("HTML version:\n", get_curator_notes_html(gid)[:300], "...\n")
print("Plain text:\n", get_curator_notes_plain(gid)[:300], "...")

HTML version:
 <i>pkaC</i> encodes the catalytic subunit of the cAMP dependent protein kinase PKA  (Mann <i>et al.</i> 1992 , Anjard <i>et al.</i> 1993). In the absence of cAMP, PKA (pkaC, pkaR)  exists as an inactive complex of the catalytic and regulatory subunits. cAMP binds cooperatively to two sites on PkaR w ...

Plain text:
 pkaC encodes the catalytic subunit of the cAMP dependent protein kinase PKA  (Mann et al. 1992 , Anjard et al. 1993). In the absence of cAMP, PKA (pkaC, pkaR)  exists as an inactive complex of the catalytic and regulatory subunits. cAMP binds cooperatively to two sites on PkaR which leads to the rel ...


In [83]:
gid = "DDB_G0283907"
tokens = get_curator_notes_tokens(gid)

claims, pubs = extract_publication_claims_from_tokens(gid, tokens)

claims_df = pl.DataFrame(claims)
pub_df = pl.DataFrame(pubs).unique(subset=["publication_id"])

claims_df.select(["gene_id","claim_plain","anchors","publication_ids"]).head(5)


gene_id,claim_plain,anchors,publication_ids
str,str,list[struct[2]],list[i64]
"""DDB_G0283907""","""pkaC encodes the catalytic sub…","[{77,[5939]}, {78,[5469]}]","[5939, 5469]"
"""DDB_G0283907""","""This protein kinase plays mult…","[{63,[5881, 4201, … 14624]}]","[5881, 4201, … 14624]"
"""DDB_G0283907""","""During normal development on a…","[{137,[14621]}]",[14621]
"""DDB_G0283907""","""The regulatory subunit can rea…","[{151,[3902]}, {152,[15133]}]","[3902, 15133]"
"""DDB_G0283907""","""As part of the circuit that ge…","[{279,[4028, 3726, 1420]}]","[4028, 3726, 1420]"


In [84]:
row0 = claims_df.row(0, named=True)
for k, v in row0.items():
    print(f"\n=== {k} ===\n{v}")



=== gene_id ===
DDB_G0283907

=== sentence_markers ===
pkaC encodes the catalytic subunit of the cAMP dependent protein kinase PKA  ([[PUB:5939]], [[PUB:5469]].

=== sentence_plain ===
pkaC encodes the catalytic subunit of the cAMP dependent protein kinase PKA  (Mann et al. 1992, Anjard et al. 1993).

=== cited_sentence_marked ===
pkaC encodes the catalytic subunit of the cAMP dependent protein kinase PKA  ([CITE:5939], [CITE:5469].

=== claim_plain ===
pkaC encodes the catalytic subunit of the cAMP dependent protein kinase PKA (,.

=== anchors ===
[{'pos': 77, 'pub_ids': [5939]}, {'pos': 78, 'pub_ids': [5469]}]

=== publication_ids ===
[5939, 5469]

=== citation_captions ===
['Mann et al. 1992', 'Anjard et al. 1993)']

=== citation_years ===
[1992, 1993]


In [81]:
# Example empty
gid = "DDB_G3946984"
note = get_curator_notes_html(gid)

if note is None:
    print(f"{gid}: no curator notes (404 or empty)")
else:
    print("HTML version:\n", note[:300], "...\n")
tokens = get_curator_notes_tokens(gid)

claims, pubs = extract_publication_claims_from_tokens(gid, tokens)

claims_df = pl.DataFrame(claims)
if pubs:  # non-empty list
    pub_df = pl.DataFrame(pubs).unique(subset=["publication_id"])
else:
    pub_df = pl.DataFrame(schema={"publication_id": pl.Int64, "caption_plain": pl.Utf8, "year": pl.Int32})

claims_df.head(), pub_df


DDB_G3946984: no curator notes (404 or empty)


(shape: (0, 0)
 ┌┐
 ╞╡
 └┘,
 shape: (0, 3)
 ┌────────────────┬───────────────┬──────┐
 │ publication_id ┆ caption_plain ┆ year │
 │ ---            ┆ ---           ┆ ---  │
 │ i64            ┆ str           ┆ i32  │
 ╞════════════════╪═══════════════╪══════╡
 └────────────────┴───────────────┴──────┘)

# get full protein list

In [34]:
# List of Reviews and associated genes (Updated monthly)
df_review = pl.read_csv("dictybase_files/Reviews.txt", separator="\t", has_header=False)
df_review.head()

column_1,column_2,column_3
str,i64,str
"""DDB_G0267374""",26183444,"""Chemotaxis/Motility, Reviews"""
"""DDB_G0267376""",26284972,"""Signal Transduction, Reviews, …"
"""DDB_G0267376""",26013485,"""Signal Transduction, Reviews, …"
"""DDB_G0267376""",18779059,"""Signal Transduction, Protein F…"
"""DDB_G0267376""",27318097,"""Signal Transduction, Reviews, …"


In [8]:
genes_review = df_review.select(df_review.columns[0]).unique()
len(genes_review)

1057

In [35]:
# DDB_G curation status (Updated monthly)
df_status = pl.read_csv("dictybase_files/DDB_G-curation_status.txt", separator="\t", has_header=False,truncate_ragged_lines=True)
df_status.head()

column_1,column_2
str,str
"""DDB_G0267212""","""Basic annotations have been ad…"
"""DDB_G0267280""","""Basic annotations have been ad…"
"""DDB_G0267304""","""Basic annotations have been ad…"
"""DDB_G0267338""","""Basic annotations have been ad…"
"""DDB_G0267356""","""Basic annotations have been ad…"


In [11]:
genes_status = df_status.select(df_status.columns[0]).unique()
len(genes_status)

18058

In [43]:
genes_status_nonempty = (
    df_status
    .filter(
        pl.col(df_status.columns[1]).is_not_null()
        & (pl.col(df_status.columns[1]) != "")
    )
    .select(pl.col(df_status.columns[0]))
    .unique()
)
len(genes_status_nonempty)

13339

In [36]:
#dictyBase ID, gene names, synonyms, and gene products (Updated monthly)
df_gene = pl.read_csv("dictybase_files/gene_information.txt", separator="\t", has_header=True)
df_gene.head()

GENE ID,Gene Name,Synonyms,Gene products
str,str,str,str
"""DDB_G0267364""","""DDB_G0267364_RTE""",null,"""Skipper GAG-PRO"""
"""DDB_G0267372""","""DDB_G0267372_RTE""",null,"""TRE5-A ORF1"""
"""DDB_G0267380""","""argE""","""P52D""","""acetylornithine deacetylase"""
"""DDB_G0267304""","""DDB_G0267304_RTE""",null,"""DIRS1 ORF3 fragment"""
"""DDB_G0267338""","""DDB_G0267338_RTE""",null,"""DIRS1 ORF3"""


In [24]:
genes_gene = df_gene.select(df_gene.columns[0]).unique()
len(genes_gene)

14222

In [37]:
#DDB-DDB_G-UniProt mapping (Updated monthly)
df_mapping = pl.read_csv("dictybase_files/DDB-GeneID-UniProt.txt", separator="\t", has_header=True)
df_mapping.head()

DDB ID,DDB_G ID,Name,UniProt ID
str,str,str,str
"""DDB0250764""","""DDB_G0267212""","""DDB_G0267212_RTE""","""Q55H52"""
"""DDB0216487""","""DDB_G0267280""","""DDB_G0267280_TE""","""Q55H16"""
"""DDB0216498""","""DDB_G0267304""","""DDB_G0267304_RTE""","""Q55H05"""
"""DDB0202373""","""DDB_G0267338""","""DDB_G0267338_RTE""","""Q55GY9"""
"""DDB0216520""","""DDB_G0267356""","""DDB_G0267356_RTE""","""Q55GX9"""


In [27]:
genes_mapping = df_mapping.select(df_mapping.columns[1]).unique()
len(genes_mapping)

14192

In [38]:
print(
    df_review.height,
    genes_status.height,
    genes_gene.height,
    genes_mapping.height,
)


3542 18058 14222 14192


In [31]:
s = pl.concat([
    genes_status.to_series(0),
    genes_gene.to_series(0),
    genes_mapping.to_series(0),
])

s.unique().len()

18058

DDB_G curation status file contains all gene id

# loop to get all curated notes

In [52]:
genes_status.head()

column_1
str
"""DDB_G3973843"""
"""DDB_G0290933"""
"""DDB_G0288271"""
"""DDB_G0276409"""
"""DDB_G0276769"""


In [54]:
def polite_sleep(base=0.2, jitter=0.10):
    time.sleep(base + random.random() * jitter)

In [60]:
rows = []
for gid in genes_status.head(20).to_series():  #<-----------------remove head for extract all
    html = None
    plain = None
    try:
        html = get_curator_notes_html(gid)
        if html:
            plain = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    except requests.RequestException as e:
        print(f"{gid}: failed ({e})")

    rows.append({
        "gene_id": gid,
        "curator_notes_html": html,
        "curator_notes_plain": plain,
    })

    polite_sleep()

DDB_G3973843: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3973843/gene/summary.json)
DDB_G3969885: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3969885/gene/summary.json)
DDB_G3971383: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3971383/gene/summary.json)
DDB_G3973327: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3973327/gene/summary.json)


In [61]:
rows

[{'gene_id': 'DDB_G3973843',
  'curator_notes_html': None,
  'curator_notes_plain': None},
 {'gene_id': 'DDB_G0290933',
  'curator_notes_html': 'Gene has been comprehensively annotated, 02-MAY-2011 PF',
  'curator_notes_plain': 'Gene has been comprehensively annotated, 02-MAY-2011 PF'},
 {'gene_id': 'DDB_G0288271',
  'curator_notes_html': 'Basic annotations have been added to this gene 4-MAY-2009 PF',
  'curator_notes_plain': 'Basic annotations have been added to this gene 4-MAY-2009 PF'},
 {'gene_id': 'DDB_G0276409',
  'curator_notes_html': 'Gene has been comprehensively annotated, 24-MAR-2011 RD',
  'curator_notes_plain': 'Gene has been comprehensively annotated, 24-MAR-2011 RD'},
 {'gene_id': 'DDB_G0276769',
  'curator_notes_html': 'A curated model has been added, 30-APR-2010 PG',
  'curator_notes_plain': 'A curated model has been added, 30-APR-2010 PG'},
 {'gene_id': 'DDB_G0289621',
  'curator_notes_html': 'A curated model has been added, 27-SEP-2010 RD',
  'curator_notes_plain': '

In [67]:
#Optional (but recommended): incremental save / resume
from pathlib import Path

OUT = Path("curator_notes.parquet")

if OUT.exists():
    df_done = pl.read_parquet(OUT)
    done_ids = set(df_done["gene_id"].to_list())
    print(f"Resuming: {len(done_ids)} genes already processed")
else:
    df_done = None
    done_ids = set()
    print("Starting fresh")

def polite_sleep(base=0.15, jitter=0.10):
    time.sleep(base + random.random() * jitter)

rows_buffer = []
BATCH_SIZE = 200   # write every 200 genes

for gid in genes_status.head(100).select("column_1").to_series():   #<-----------------remove head for extract all
    if gid in done_ids:
        continue   # ← THIS enables resume

    try:
        html = get_curator_notes_html(gid)
        plain = (
            BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
            if html else None
        )
    except Exception as e:
        print(f"{gid}: failed ({e})")
        html = None
        plain = None

    rows_buffer.append({
        "gene_id": gid,
        "curator_notes_html": html,
        "curator_notes_plain": plain,
    })

    # periodically flush to disk
    if len(rows_buffer) >= BATCH_SIZE:
        df_batch = pl.DataFrame(rows_buffer)

        if OUT.exists():
            df_batch.write_parquet(OUT, append=True)
        else:
            df_batch.write_parquet(OUT)

        rows_buffer.clear()

    polite_sleep()
    
if rows_buffer:
    df_batch = pl.DataFrame(rows_buffer)
    if OUT.exists():
        df_batch.write_parquet(OUT, append=True)
    else:
        df_batch.write_parquet(OUT)

Starting fresh
DDB_G3973843: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3973843/gene/summary.json)
DDB_G3969885: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3969885/gene/summary.json)
DDB_G3971383: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3971383/gene/summary.json)
DDB_G3973327: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3973327/gene/summary.json)
DDB_G3971993: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3971993/gene/summary.json)
DDB_G0290665: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G0290665/gene/summary.json)
DDB_G3972953: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G3972953/gene/summary.json)
DDB_G0279847: failed (500 Server Error:  for url: http://dictybase.org/gene/DDB_G0279847/gene/summary.json)


In [72]:
df = pl.read_parquet("curator_notes.parquet")
df

gene_id,curator_notes_html,curator_notes_plain
str,str,str
"""DDB_G3965802""",null,null
"""DDB_G0275703""","""<i>gefN</i> encodes a Ras guan…","""gefN encodes a Ras guanine nuc…"
"""DDB_G0282987""","""Basic annotations have been ad…","""Basic annotations have been ad…"
"""DDB_G0268874""","""A curated model has been added…","""A curated model has been added…"
"""DDB_G3949950""",null,null
"""DDB_G3970901""",null,null
"""DDB_G3965980""",null,null
"""DDB_G0293360""","""Gene has been comprehensively …","""Gene has been comprehensively …"
"""DDB_G0292826""","""Basic annotations have been ad…","""Basic annotations have been ad…"


# full results

command: docker run -it -v "$PWD/output:/dictycite/output" --platform=linux/amd64 fulaibaowang/dictycite:22.12.2025 python dicty_curator_notes.py --limit 0 --sleep-base 0.25 --sleep-jitter 0.10

takes around 2 hrs

In [86]:
df = pl.read_parquet("output/curator_notes.parquet")
df

gene_id,curator_notes_html,curator_notes_plain
str,str,str
"""DDB_G0290911""","""A curated model has been added…","""A curated model has been added…"
"""DDB_G0288243""","""A curated model has been added…","""A curated model has been added…"
"""DDB_G0288327""","""Basic annotations have been ad…","""Basic annotations have been ad…"
"""DDB_G0273271""","""Basic annotations have been ad…","""Basic annotations have been ad…"
"""DDB_G0279339""","""A curated model has been added…","""A curated model has been added…"
…,…,…
"""DDB_G3960442""",null,null
"""DDB_G0284811""","""A curated model has been added…","""A curated model has been added…"
"""DDB_G0294609""","""Gene has been comprehensively …","""Gene has been comprehensively …"


In [88]:
subset = df.filter(
    pl.col("curator_notes_html")
      .fill_null("")
      .str.contains("<i>")
)
subset

gene_id,curator_notes_html,curator_notes_plain
str,str,str
"""DDB_G0282143""","""<i>hatB</i> encodes the histid…","""hatB encodes the histidine-ric…"
"""DDB_G0293544""","""The <i>cepK</i> gene (CP250) w…","""The cepK gene (CP250) was iden…"
"""DDB_G0290439""","""<i>gacW</i> (CARMIL-GAP) encod…","""gacW (CARMIL-GAP) encodes a Ra…"
"""DDB_G0275323""","""<i>tipD</i> was isolated from …","""tipD was isolated from a REMI …"
"""DDB_G0269026""","""The ABC superfamily of genes i…","""The ABC superfamily of genes i…"
…,…,…
"""DDB_G0273885""","""Papers from the early '80's su…","""Papers from the early '80's su…"
"""DDB_G0275029""","""The 2-oxoglutarate dehydrogena…","""The 2-oxoglutarate dehydrogena…"
"""DDB_G0269298""","""<i>gefX</i> encodes a protein …","""gefX encodes a protein with an…"


In [8]:
subset.filter(
    pl.col("gene_id") == "DDB_G0283907"
).select("curator_notes_html").to_series()[0]


'<i>pkaC</i> encodes the catalytic subunit of the cAMP dependent protein kinase PKA  (Mann <i>et al.</i> 1992 , Anjard <i>et al.</i> 1993). In the absence of cAMP, PKA (pkaC, pkaR)  exists as an inactive complex of the catalytic and regulatory subunits. cAMP binds cooperatively to two sites on PkaR which leads to the release of active PkaC. This protein kinase plays multiple roles throughout development (Harwood <i>et al.</i> 1992, Mann <i>et al.</i> 1997, Loomis <i>et al.</i> 1998, Loomis <i>et al.</i> 2015). During normal development on a moist surface, mRNA from <i>pkaC</i> increases smoothly from an initially low level to reach a peak at 20 hours (Rosengarten <i>et al.</i> 2015). When cells are developed in suspension with addition of 50 nM cAMP every 6 minutes,  the level of <i>pkaC</i> mRNA stays low throughout development. Very few other genes are repressed by cAMP pulses. The regulatory subunit can reassociate with the catalytic subunit and inhibit it when the phosphodiesterase

## summarize the year

In [6]:
year_pat = r"\b(18\d{2}|19\d{2}|20\d{2})\b"

year_counts = (
    subset
    .select(
        pl.col("curator_notes_html")
          .fill_null("")
          .str.extract_all(year_pat)   # -> list[str] of years per row
          .alias("years")
    )
    .explode("years")                 # one year per row
    .filter(pl.col("years").is_not_null() & (pl.col("years") != ""))
    .with_columns(pl.col("years").cast(pl.Int32).alias("year"))
    .group_by("year")
    .agg(pl.len().alias("n"))         # occurrences (mentions), not unique
    .sort("year")
)

year_counts

year,n
i32,u32
1823,1
1833,1
1956,1
1965,1
1967,3
…,…
2020,48
2021,32
2022,32


In [7]:
hits_2029 = (
    subset
    .filter(
        pl.col("curator_notes_html").fill_null("").str.contains(r"\b2029\b")
        # or use curator_notes_plain if you counted on plain
    )
    .select(["gene_id", "curator_notes_html", "curator_notes_plain"])
)

hits_2029[0,2]


'MHCK A ( mhkA ) was the first myosin heavy chain kinase identified, purified as a 130 kDa protein. It was found to phophorylate myosin II on the heavy chain (MHC, mhcA )  driving myosin filament disassembly  (Cote & Bukiejko 1987). In this study MHCK A was found to phosphorylate threonine residues, later these threonine residues were identified to be at positions 1823, 1833, and 2029 in the tail region of myosin II  (Vaillancourt et al. 1988),  (Luck-Vielmetter et al. 1999). It was shown that MHCK A activity is activated by autophosphorylation, and that the presence of myosin II increased the rate of autophosphorylation  (Medley et al. 1990). Furthermore it was shown that MHCK A has a high preference for threonine;  Luo et al. 2001,  Crawley & Cote et al. 2008)) and neighboring peptide residues important for substrate specificity have been determined ( Crawley & Cote et al. 2008). In 1995, Futey et al. cloned the mhkA gene encoding MHCK A and found that no part of the sequence display

In [ ]:
claims

In [90]:
claims = pl.read_parquet("output/curator_claims.parquet")
claims

gene_id,sentence_markers,sentence_plain,cited_sentence_marked,claim_plain,anchors,publication_ids,citation_captions,citation_years
str,str,str,str,str,list[struct[2]],list[i64],list[str],list[i64]
"""DDB_G0282143""","""Hisactophilin II is distribute…","""Hisactophilin II is distribute…","""Hisactophilin II is distribute…","""Hisactophilin II is distribute…","[{142,[5032]}]",[5032],"[""Hanakam et al. 1995""]",[1995]
"""DDB_G0282143""","""Upon acidification, hatB overe…","""Upon acidification, hatB overe…","""Upon acidification, hatB overe…","""Upon acidification, hatB overe…","[{125,[4561]}]",[4561],"[""Stoeckelhuber et al. 1996""]",[1996]
"""DDB_G0282143""","""In addition, hisactophilin-def…","""In addition, hisactophilin-def…","""In addition, hisactophilin-def…","""In addition, hisactophilin-def…","[{120,[2166]}]",[2166],"[""Pintsch et al. 2002""]",[2002]
"""DDB_G0275323""","""tipD was isolated from a REMI …","""tipD was isolated from a REMI …","""tipD was isolated from a REMI …","""tipD was isolated from a REMI …","[{131,[3448]}]",[3448],"[""Stege et al. 1999)""]",[1999]
"""DDB_G0275323""","""The product of tipD has been s…","""The product of tipD has been s…","""The product of tipD has been s…","""The product of tipD has been s…","[{91,[16104]}]",[16104],"[""Mesquita et al. 2015)""]",[2015]
…,…,…,…,…,…,…,…,…
"""DDB_G0273885""","""Plant lectins induce the same …","""Plant lectins induce the same …","""Plant lectins induce the same …","""Plant lectins induce the same …","[{215,[17611]}]",[17611],"[""(Dinh et al. 2018)""]",[2018]
"""DDB_G0269298""","""gefX is expressed during both …","""gefX is expressed during both …","""gefX is expressed during both …","""gefX is expressed during both …","[{127,[1231]}]",[1231],"[""Wilkins et al. 2005)""]",[2005]
"""DDB_G0276019""","""gefY is expressed during both …","""gefY is expressed during both …","""gefY is expressed during both …","""gefY is expressed during both …","[{127,[1231]}]",[1231],"[""Wilkins et al. 2005)""]",[2005]


In [91]:
publications = pl.read_parquet("output/publications.parquet")
publications

publication_id,caption_plain,year
i64,str,i64
5032,"""Hanakam et al. 1995""",1995
4561,"""Stoeckelhuber et al. 1996""",1996
2166,"""Pintsch et al. 2002""",2002
3448,"""Stege et al. 1999)""",1999
16104,"""Mesquita et al. 2015)""",2015
…,…,…
17611,"""(Dinh et al. 2018)""",2018
1231,"""Wilkins et al. 2005)""",2005
1231,"""Wilkins et al. 2005)""",2005
